#1.1)CODIGO FUNCIONAL PARA ABRIR Y CERRAR PESTAÑAS-NEW CODIGO BASE

#Perfecto para DESCONEXION MANUAL

In [2]:
import os
import time
import logging
import pandas as pd
import threading
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions

# Configuración de logging
logging.basicConfig(
    filename='registro_errores.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

class AutomatizacionGemas:
    def __init__(self):
        self.df = None
        self.procesando = False
        self.resultados_proceso = []
        self.driver = None
        
        # Rutas específicas
        self.excel_path = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\Insertar_Data_Excel1.xlsx"
        self.edgedriver_path = os.path.join(os.getcwd(), "msedgedriver.exe")
        
        # Validar el driver
        if not os.path.isfile(self.edgedriver_path):
            print(f"⚠️ ADVERTENCIA: msedgedriver.exe no encontrado en: {self.edgedriver_path}")
            raise FileNotFoundError(f"msedgedriver.exe no encontrado en: {self.edgedriver_path}")

    def imprimir_log(self, mensaje):
        """Imprime un mensaje con timestamp"""
        timestamp = time.strftime("%H:%M:%S")
        print(f"[{timestamp}] {mensaje}")
        
    def inicializar_navegador(self):
        """Inicializa y configura el navegador Edge"""
        try:
            edge_options = EdgeOptions()
            edge_options.add_argument("--start-maximized")
            edge_options.add_argument("--disable-blink-features=AutomationControlled")
            edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
            edge_options.add_experimental_option('useAutomationExtension', False)
            
            # Usar perfil persistente para mantener sesión
            edge_options.add_argument("--user-data-dir=C:\\Temp\\EdgeAutomationProfile")
            
            service = EdgeService(executable_path=self.edgedriver_path)
            self.driver = webdriver.Edge(service=service, options=edge_options)
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            
            self.imprimir_log("✅ Navegador Edge inicializado correctamente")
            return True
            
        except Exception as e:
            self.imprimir_log(f"❌ Error al inicializar navegador: {str(e)}")
            return False

    def cerrar_ventanas_emergentes(self, wait, max_intentos=6):
        """Cierra ventanas emergentes si existen"""
        for intento in range(max_intentos):
            try:
                ventana = wait.until(EC.presence_of_element_located(
                    (By.XPATH, '//div[@class="modal-dialog"]')
                ))
                boton_cerrar = ventana.find_element(By.XPATH, './/button[normalize-space()="Cerrar"]')
                boton_cerrar.click()
                time.sleep(1)
                self.imprimir_log(f"   Ventana emergente cerrada (intento {intento + 1})")
            except:
                break

    def rellenar_condicional(self, xpath_objetivo, xpath_fuente, wait):
        """Rellena un campo solo si está vacío"""
        try:
            campo = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, xpath_objetivo))
            )
            if campo.get_attribute("value").strip() == "":
                fuente = self.driver.find_element(By.XPATH, xpath_fuente)
                valor = fuente.get_attribute("value")
                campo.send_keys(valor)
                time.sleep(1)
                self.cerrar_ventanas_emergentes(wait)
                self.imprimir_log(f"   Campo {xpath_objetivo} rellenado con valor de {xpath_fuente}")
        except Exception as e:
            self.imprimir_log(f"   ⚠️ No se pudo rellenar campo condicional: {str(e)}")

    def intentar_click(self, xpath, timeout=4, descripcion=""):
        """Intenta hacer clic en un elemento con manejo de errores"""
        try:
            elemento = WebDriverWait(self.driver, timeout).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            elemento.click()
            time.sleep(1)
            if descripcion:
                self.imprimir_log(f"   {descripcion}")
            return True
        except (TimeoutException, NoSuchElementException):
            return False

    def scroll_hasta_elemento(self, xpath_elemento, xpath_scrollbar="//*[@id='ascrail2004']/div"):
        """Desplaza el scrollbar hasta hacer visible un elemento específico"""
        try:
            # Primero intentamos encontrar el elemento
            elemento = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, xpath_elemento))
            )
            
            # Si ya es visible, no necesitamos hacer scroll
            if elemento.is_displayed():
                self.imprimir_log(f"   Elemento ya visible: {xpath_elemento}")
                return True
            
            # Intentar hacer scroll usando JavaScript
            self.driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", elemento)
            time.sleep(2)
            
            # Si hay scrollbar específico, intentamos moverlo
            try:
                scrollbar = self.driver.find_element(By.XPATH, xpath_scrollbar)
                self.driver.execute_script("arguments[0].scrollTop += 300;", scrollbar)
                time.sleep(1)
            except:
                pass
                
            # Verificar si ahora es visible
            if elemento.is_displayed():
                self.imprimir_log(f"   Elemento hecho visible después de scroll: {xpath_elemento}")
                return True
            else:
                self.imprimir_log(f"   ⚠️ Elemento aún no visible después de scroll: {xpath_elemento}")
                return False
                
        except Exception as e:
            self.imprimir_log(f"   ⚠️ Error al hacer scroll hasta elemento: {str(e)}")
            return False

    def hacer_doble_click(self, xpath_elemento):
        """Realiza doble clic en un elemento"""
        try:
            elemento = WebDriverWait(self.driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, xpath_elemento))
            )
            
            # Usar JavaScript para doble clic
            self.driver.execute_script("""
                var event = new MouseEvent('dblclick', {
                    view: window,
                    bubbles: true,
                    cancelable: true
                });
                arguments[0].dispatchEvent(event);
            """, elemento)
            
            time.sleep(2)
            self.imprimir_log(f"   Doble clic realizado en: {xpath_elemento}")
            return True
            
        except Exception as e:
            self.imprimir_log(f"   ⚠️ Error al hacer doble clic: {str(e)}")
            return False

    def verificar_url_contiene(self, texto):
        """Verifica si la URL actual contiene un texto específico"""
        try:
            current_url = self.driver.current_url
            if texto in current_url:
                self.imprimir_log(f"   URL actual contiene '{texto}': {current_url}")
                return True
            else:
                self.imprimir_log(f"   URL actual NO contiene '{texto}': {current_url}")
                return False
        except Exception as e:
            self.imprimir_log(f"   ⚠️ Error al verificar URL: {str(e)}")
            return False

    def agregar_elemento_danado(self, wait, subestacion, parque, lmt):
        """Maneja el proceso de agregar elemento dañado con scroll y validaciones"""
        try:
            # 1. Primero hacer clic en el botón para agregar elemento dañado
            self.imprimir_log("   Intentando agregar elemento dañado...")
            
            boton_encontrado = False
            rutas_botones = [
                '/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[2]/ul/li[2]/button/div/span',
                '/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[2]/ul/li[2]/button/div',
                '//button[contains(@class, "btn") and contains(., "Agregar")]',
                '//button[contains(., "Elemento")]',
                '//button[contains(., "dañado")]'
            ]
            
            for ruta in rutas_botones:
                try:
                    boton = WebDriverWait(self.driver, 3).until(
                        EC.element_to_be_clickable((By.XPATH, ruta))
                    )
                    boton.click()
                    self.imprimir_log(f"   Botón de agregar elemento dañado clickeado: {ruta}")
                    boton_encontrado = True
                    time.sleep(3)
                    break
                except:
                    continue
            
            if not boton_encontrado:
                self.imprimir_log("   ⚠️ No se pudo encontrar el botón de agregar elemento dañado")
                return False
            
            # 2. Verificar que estamos en la URL correcta

            
            self.imprimir_log("   Haciendo scroll hasta el textarea...")
            scroll_exitoso = self.scroll_hasta_elemento(xpath_textarea, xpath_scrollbar)
            
            if not scroll_exitoso:
                # Intentar método alternativo de scroll
                self.imprimir_log("   Intentando método alternativo de scroll...")
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.7);")
                time.sleep(2)
            
            # 4. Hacer doble clic en el textarea
            self.imprimir_log("   Intentando hacer doble clic en el textarea...")
            doble_click_exitoso = self.hacer_doble_click(xpath_textarea)
            
            if doble_click_exitoso:
                self.imprimir_log("   ✅ Doble clic exitoso en textarea")
            else:
                self.imprimir_log("   ⚠️ No se pudo hacer doble clic, intentando clic normal...")
                self.intentar_click(xpath_textarea, descripcion="Clic en textarea")
            
            # 5. Esperar a que aparezcan los campos de formulario
            time.sleep(3)
            
            # Verificar si estamos en el formulario correcto
            try:
                WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.XPATH, '//*[@id="damagedElement"]'))
                )
                self.imprimir_log("   ✅ Formulario de elemento dañado cargado")
            except:
                self.imprimir_log("   ⚠️ Formulario de elemento dañado no encontrado")
            
            # 6. Rellenar campos de elemento dañado
            campos_danados = [
                ('//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[2]/input', 
                 subestacion, "Subestación"),
                ('//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[4]/input', 
                 str(parque), "PARQUE"),
                ('//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[5]/input', 
                 lmt, "LMT"),
            ]

            for xpath, valor, nombre in campos_danados:
                try:
                    # Esperar a que el campo sea interactuable
                    campo = wait.until(
                        EC.element_to_be_clickable((By.XPATH, xpath))
                    )
                    
                    # Limpiar y rellenar
                    campo.clear()
                    campo.send_keys(valor)
                    time.sleep(1)
                    
                    # Presionar ENTER para activar el autocompletado
                    campo.send_keys(Keys.ENTER)
                    time.sleep(2)
                    
                    self.imprimir_log(f"   Campo {nombre} rellenado: {valor}")
                    
                except Exception as e:
                    self.imprimir_log(f"   ⚠️ No se pudo rellenar {nombre}: {str(e)}")
                    # Intentar método alternativo
                    try:
                        campo_js = self.driver.find_element(By.XPATH, xpath)
                        self.driver.execute_script("arguments[0].value = arguments[1];", campo_js, valor)
                        time.sleep(1)
                        self.imprimir_log(f"   Campo {nombre} rellenado con JavaScript")
                    except:
                        self.imprimir_log(f"   ❌ Error crítico rellenando {nombre}")

            # 7. Procesar campo "9" con dropdown
            try:
                fld9 = wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[9]/input'))
                )
                fld9.clear()
                fld9.send_keys("9")
                time.sleep(2)
                
                # Esperar a que aparezcan las opciones del dropdown
                time.sleep(2)
                
                # Buscar opciones del dropdown
                all_opts = wait.until(
                    EC.presence_of_all_elements_located((By.XPATH, "//ul[contains(@class,'dropdown-menu')]//a[@title]"))
                )
                
                visible_opts = [opt for opt in all_opts if opt.is_displayed()]
                priorities = ["9", "1", "9F"]
                clicked = False

                for title in priorities:
                    for opt in visible_opts:
                        if opt.get_attribute("title") == title:
                            opt.click()
                            clicked = True
                            self.imprimir_log(f"   Opción seleccionada: {title}")
                            time.sleep(2)
                            break
                    if clicked:
                        break

                if not clicked and visible_opts:
                    # Seleccionar la primera opción visible
                    visible_opts[0].click()
                    self.imprimir_log("   Primera opción visible seleccionada")
                    
            except Exception as e:
                self.imprimir_log(f"   ⚠️ No se pudo procesar campo '9': {str(e)}")
                # Continuar de todos modos

            # 8. Seleccionar fila si existe
            try:
                WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "div.ui-grid-selection-row-header-buttons.ui-grid-icon-ok"))
                ).click()
                self.imprimir_log("   Fila seleccionada")
                time.sleep(1)
            except:
                try:
                    # Intentar otro selector
                    WebDriverWait(self.driver, 3).until(
                        EC.element_to_be_clickable((By.XPATH, "//div[contains(@class, 'ui-grid-row-selected')]"))
                    ).click()
                    self.imprimir_log("   Fila seleccionada (selector alternativo)")
                except:
                    self.imprimir_log("   ⚠️ No se pudo seleccionar fila")

            # 9. Confirmaciones finales
            for i in range(2):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button[1]', 
                                  descripcion=f"Confirmación final {i+1}.1")
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', 
                                  descripcion=f"Confirmación final {i+1}.2")
                time.sleep(1)
            
            self.imprimir_log("   ✅ Proceso de agregar elemento dañado completado")
            return True
            
        except Exception as e:
            self.imprimir_log(f"   ❌ Error en agregar elemento dañado: {str(e)}")
            return False

    def procesar_registro(self, index, row, wait):
        """Procesa un registro individual en una nueva pestaña"""
        ref_cc = str(row["REF CC"])
        subestacion = str(row["Subestación"])
        parque = int(row["PARQUE"])
        lmt = str(row["LMT"])

        self.imprimir_log(f"📝 Procesando registro {index + 1}: REF_CC={ref_cc}")
        
        try:
            # Abrir nueva pestaña
            self.driver.execute_script("window.open('');")
            self.driver.switch_to.window(self.driver.window_handles[-1])
            
            # INICIO LÓGICA SELENIUM EN NUEVA PESTAÑA
            self.driver.get("http://gemas-pe.enelint.global")
            time.sleep(3)

            # Navegación principal
            try:
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, "/html/body/div[3]/div/div[2]/div/div[1]/div/a/img"))
                ).click()
                time.sleep(2)

                # Buscar por REF_CC
                input_element = wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="test"]/div[1]/div[1]/div[1]/div[2]/input'))
                )
                input_element.clear()
                input_element.send_keys(ref_cc)
                time.sleep(2)
                
                # Clic en buscar
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, "/html/body/div[3]/div/div/div[2]/div/toolbar/div/div[3]/ul/li[1]/button"))
                ).click()
                time.sleep(2)

                # Seleccionar resultado
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[contains(@id, "-grid-container")]/div[2]/div'))
                ).click()
                time.sleep(2)

                # Abrir menú
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="menu"]/ul/li[1]/div/div[1]/a'))
                ).click()
                time.sleep(2)
                
            except Exception as e:
                raise Exception(f"Error en navegación inicial: {str(e)}")

            # Rellenar campos condicionales
            self.rellenar_condicional('//*[@id="beginRepairDate"]', '//*[@id="beginDate"]', wait)
            self.rellenar_condicional('//*[@id="localizationDate"]', '//*[@id="beginDate"]', wait)
            self.rellenar_condicional('//*[@id="endRepairDate"]', '//*[@id="endDate"]', wait)
            self.rellenar_condicional('//*[@id="estResolutionDate"]', '//*[@id="endDate"]', wait)

            # Intentar guardar múltiples veces
            for i in range(6):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', descripcion=f"Guardar {i + 1}")

            # Navegar entre pestañas y rellenar
            wait.until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[3]/div/div[2]/div/div/div/div/form/div[3]/div[2]/div/div/ul/li[1]/a'))
            ).click()
            time.sleep(2)

            for i in range(3):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button')

            # Rellenar CC
            cc = self.driver.find_element(By.XPATH, '//*[@id="textAreaCC"]/textarea')
            if not cc.get_attribute("value").strip():
                cc.send_keys(".")
            time.sleep(1)

            # Continuar con UOT
            wait.until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[3]/div/div[2]/div/div/div/div/form/div[3]/div[2]/div/div/ul/li[2]/a'))
            ).click()
            time.sleep(2)

            # Rellenar UOT
            uot = self.driver.find_element(By.XPATH, '//*[@id="textAreaUOT"]/textarea')
            if not uot.get_attribute("value").strip():
                uot.send_keys(".")
            time.sleep(1)

            # Continuar con CDS
            wait.until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[3]/div/div[2]/div/div/div/div/form/div[3]/div[2]/div/div/ul/li[3]/a'))
            ).click()
            time.sleep(2)

            # Rellenar CDS
            cds = self.driver.find_element(By.XPATH, '//*[@id="textAreaCds"]/textarea')
            if not cds.get_attribute("value").strip():
                cds.send_keys("VAut")
            time.sleep(1)

            # Guardar final
            self.intentar_click('/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[2]/ul/li[1]/button/div/img', descripcion="Guardar final")
            time.sleep(6)

            # Manejo de ventanas de confirmación
            self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button[1]', descripcion="Confirmación 1")
            time.sleep(3)
            
            self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', descripcion="Confirmación 2")
            time.sleep(3)

            # AGREGAR ELEMENTO DAÑADO CON SCROLL Y DOBLE CLIC
            elemento_agregado = self.agregar_elemento_danado(wait, subestacion, parque, lmt)
            
            if not elemento_agregado:
                self.imprimir_log("   ⚠️ Se saltó el proceso de agregar elemento dañado debido a errores")

            # Finalizar proceso
            self.intentar_click('//*[@id="menu"]/ul/li[1]/div/div[1]/a', descripcion="Volver al menú")
            time.sleep(2)
            
            self.intentar_click('/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[3]/ul/li[2]/button', descripcion="Finalizar")
            time.sleep(2)

            # Confirmaciones finales múltiples
            for i in range(8):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', descripcion=f"Confirmación extra {i+1}")
                time.sleep(0.5)

            # Registrar éxito
            self.resultados_proceso.append({
                "Fila": index + 1,
                "REF CC": ref_cc,
                "Subestación": subestacion,
                "PARQUE": parque,
                "LMT": lmt,
                "Estado": "✅ Correcto",
                "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "Empresa": "PLUZ ENERGÍA",
                "Elemento_Danado": "✅ Agregado" if elemento_agregado else "⚠️ No agregado"
            })
            
            self.imprimir_log(f"✅ Registro {index + 1} procesado correctamente - PLUZ")
            return True
            
        except Exception as e:
            error_msg = f"❌ Error en registro {index + 1}: {str(e)}"
            logging.error(error_msg, exc_info=True)
            self.resultados_proceso.append({
                "Fila": index + 1,
                "REF CC": ref_cc,
                "Subestación": subestacion,
                "PARQUE": parque,
                "LMT": lmt,
                "Estado": f"❌ Error: {str(e)}",
                "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "Empresa": "PLUZ ENERGÍA",
                "Elemento_Danado": "❌ Error"
            })
            self.imprimir_log(error_msg)
            return False
            
        finally:
            # Cerrar pestaña actual y volver a la principal
            try:
                if len(self.driver.window_handles) > 1:
                    self.driver.close()
                    self.driver.switch_to.window(self.driver.window_handles[0])
            except:
                pass

    # ... (resto de métodos permanecen igual: ejecutar_procesamiento, exportar_resultados, etc.)

    def ejecutar_procesamiento(self):
        """Ejecuta el procesamiento principal"""
        try:
            # Inicializar navegador
            if not self.inicializar_navegador():
                return

            wait = WebDriverWait(self.driver, 20)

            # FASE 1: Login manual
            self.imprimir_log("🔐 FASE 1: Preparando para ingreso manual de credenciales...")
            self.imprimir_log("📝 Por favor, ingresa manualmente tu usuario y contraseña")
            self.imprimir_log("⏳ Esperando 30 segundos para ingreso manual...")
            
            # Abrir página de login
            self.driver.get("http://gemas-pe.enelint.global")
            time.sleep(40)
            
            # Verificar login
            try:
                # Esperar a que cambie la página después del login
                time.sleep(5)
                current_url = self.driver.current_url
                if "login" not in current_url.lower():
                    self.imprimir_log("✅ Login manual detectado - Continuando...")
                else:
                    self.imprimir_log("⚠️ Parece que aún estás en la página de login")
            except:
                self.imprimir_log("⚠️ Continuando con procesamiento...")

            # FASE 2: Procesamiento
            self.imprimir_log("🚀 FASE 2: Iniciando procesamiento automatizado...")
            
            for index, row in self.df.iterrows():
                if not self.procesando:
                    break
                    
                self.procesar_registro(index, row, wait)
                
                # Pequeña pausa entre registros
                time.sleep(2)

            # Finalizar
            if self.procesando:
                self.imprimir_log("🎉 AUTOMATIZACIÓN COMPLETADA EXITOSAMENTE")
                self.imprimir_log("⚡ PLUZ ENERGÍA - MISIÓN CUMPLIDA ⚡")
                self.exportar_resultados()
                
        except Exception as e:
            logging.error("Error general durante la ejecución", exc_info=True)
            self.imprimir_log(f"❌ Error general: {str(e)}")
        
        finally:
            self.finalizar_procesamiento()

    def exportar_resultados(self):
        """Exporta los resultados a un archivo Excel"""
        try:
            if not self.resultados_proceso:
                self.imprimir_log("⚠️ No hay resultados para exportar")
                return
            
            df_resultados = pd.DataFrame(self.resultados_proceso)
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_path = f"resultados_proceso_{timestamp}.xlsx"
            
            df_resultados.to_excel(output_path, index=False)
            self.imprimir_log(f"📊 Resultados exportados a: {output_path}")
            
            # Resumen detallado
            correctos = len([r for r in self.resultados_proceso if 'Correcto' in str(r['Estado'])])
            errores = len([r for r in self.resultados_proceso if 'Error' in str(r['Estado'])])
            elemento_agregado = len([r for r in self.resultados_proceso if 'Agregado' in str(r.get('Elemento_Danado', ''))])
            
            self.imprimir_log(f"📈 Resumen:")
            self.imprimir_log(f"   ✅ Correctos: {correctos}")
            self.imprimir_log(f"   ❌ Errores: {errores}")
            self.imprimir_log(f"   🔧 Elemento dañado agregado: {elemento_agregado}")
            
        except Exception as e:
            self.imprimir_log(f"❌ Error al exportar resultados: {str(e)}")

    def finalizar_procesamiento(self):
        """Finaliza el procesamiento y cierra recursos"""
        try:
            if self.driver:
                self.driver.quit()
                self.imprimir_log("✅ Navegador cerrado")
        except:
            pass
        
        self.procesando = False
        self.imprimir_log("🏁 Procesamiento finalizado")

    def iniciar(self):
        """Método principal para iniciar la automatización"""
        try:
            # Cargar el archivo Excel
            if not os.path.exists(self.excel_path):
                print(f"❌ Error: Archivo Excel no encontrado en {self.excel_path}")
                return
            
            self.df = pd.read_excel(self.excel_path)
            total_filas = len(self.df)
            
            if total_filas == 0:
                print("❌ Error: El archivo Excel está vacío")
                return
            
            print(f"📊 Archivo Excel cargado: {total_filas} registros")
            print("=" * 60)
            
            # Mostrar resumen de columnas
            print("Columnas encontradas:")
            for col in self.df.columns:
                print(f"  - {col}")
            print("=" * 60)
            
            # Confirmar inicio
            respuesta = input("¿Desea iniciar el procesamiento? (s/n): ").lower()
            if respuesta != 's':
                print("Proceso cancelado")
                return
            
            self.procesando = True
            self.resultados_proceso = []
            
            # Ejecutar procesamiento
            self.ejecutar_procesamiento()
            
        except Exception as e:
            print(f"❌ Error al iniciar: {str(e)}")
        finally:
            input("\nPresione Enter para salir...")


# Ejecutar la automatización
if __name__ == "__main__":
    app = AutomatizacionGemas()
    app.iniciar()

📊 Archivo Excel cargado: 64 registros
Columnas encontradas:
  - DIA
  - REF CC
  - NT
  - UOT
  - CFT
  - PARQUE
  - Subestación
  - LMT
  - TIPO
  - CAUSA
[14:14:23] ✅ Navegador Edge inicializado correctamente
[14:14:23] 🔐 FASE 1: Preparando para ingreso manual de credenciales...
[14:14:23] 📝 Por favor, ingresa manualmente tu usuario y contraseña
[14:14:23] ⏳ Esperando 30 segundos para ingreso manual...
[14:15:18] ✅ Login manual detectado - Continuando...
[14:15:18] 🚀 FASE 2: Iniciando procesamiento automatizado...
[14:15:18] 📝 Procesando registro 1: REF_CC=B069223
[14:15:34]    Campo //*[@id="beginRepairDate"] rellenado con valor de //*[@id="beginDate"]
[14:15:37]    Guardar 1
[14:15:40]    Guardar 2
[14:15:41]    Guardar 3
[14:15:42]    Guardar 4
[14:15:44]    Guardar 5
[14:16:08]    Guardar final
[14:16:15]    Confirmación 1
[14:16:19]    Confirmación 2
[14:16:22]    Intentando agregar elemento dañado...
[14:16:22]    Botón de agregar elemento dañado clickeado: /html/body/div[3]/di

KeyboardInterrupt: 

In [ ]:
#PRUEBE PARA AÑADIR ELEMENTO AVERIADO (FUNCIONA, PROBLEMAS EN EL TIEMPO DE ESPERA)

In [ ]:
import os
import time
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.common.action_chains import ActionChains  # Importar ActionChains

# Configuración de logging
logging.basicConfig(
    filename='registro_errores.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

class AutomatizacionGemas:
    def __init__(self):
        self.df = None
        self.procesando = False
        self.resultados_proceso = []
        self.driver = None
        
        # Rutas específicas
        self.excel_path = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\Insertar_Data_Excel.xlsx"
        self.edgedriver_path = os.path.join(os.getcwd(), "msedgedriver.exe")
        
        # Validar el driver
        if not os.path.isfile(self.edgedriver_path):
            print(f"⚠️ ADVERTENCIA: msedgedriver.exe no encontrado en: {self.edgedriver_path}")
            raise FileNotFoundError(f"msedgedriver.exe no encontrado en: {self.edgedriver_path}")

    def imprimir_log(self, mensaje):
        """Imprime un mensaje con timestamp"""
        timestamp = time.strftime("%H:%M:%S")
        print(f"[{timestamp}] {mensaje}")
        
    def inicializar_navegador(self):
        """Inicializa y configura el navegador Edge"""
        try:
            edge_options = EdgeOptions()
            edge_options.add_argument("--start-maximized")
            edge_options.add_argument("--disable-blink-features=AutomationControlled")
            edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
            edge_options.add_experimental_option('useAutomationExtension', False)
            
            # Usar perfil persistente para mantener sesión
            edge_options.add_argument("--user-data-dir=C:\\Temp\\EdgeAutomationProfile")
            
            service = EdgeService(executable_path=self.edgedriver_path)
            self.driver = webdriver.Edge(service=service, options=edge_options)
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            
            self.imprimir_log("✅ Navegador Edge inicializado correctamente")
            return True
            
        except Exception as e:
            self.imprimir_log(f"❌ Error al inicializar navegador: {str(e)}")
            return False

    def cerrar_ventanas_emergentes(self, wait, max_intentos=6):
        """Cierra ventanas emergentes si existen"""
        for intento in range(max_intentos):
            try:
                ventana = wait.until(EC.presence_of_element_located(
                    (By.XPATH, '//div[@class="modal-dialog"]')
                ))
                boton_cerrar = ventana.find_element(By.XPATH, './/button[normalize-space()="Cerrar"]')
                boton_cerrar.click()
                time.sleep(1)
                self.imprimir_log(f"   Ventana emergente cerrada (intento {intento + 1})")
            except:
                break

    def rellenar_condicional(self, xpath_objetivo, xpath_fuente, wait):
        """Rellena un campo solo si está vacío"""
        try:
            campo = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, xpath_objetivo))
            )
            if campo.get_attribute("value").strip() == "":
                fuente = self.driver.find_element(By.XPATH, xpath_fuente)
                valor = fuente.get_attribute("value")
                campo.send_keys(valor)
                time.sleep(1)
                self.cerrar_ventanas_emergentes(wait)
                self.imprimir_log(f"   Campo {xpath_objetivo} rellenado con valor de {xpath_fuente}")
        except Exception as e:
            self.imprimir_log(f"   ⚠️ No se pudo rellenar campo condicional: {str(e)}")

    def intentar_click(self, xpath, timeout=4, descripcion=""):
        """Intenta hacer clic en un elemento con manejo de errores"""
        try:
            elemento = WebDriverWait(self.driver, timeout).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            elemento.click()
            time.sleep(1)
            if descripcion:
                self.imprimir_log(f"   {descripcion}")
            return True
        except (TimeoutException, NoSuchElementException):
            return False

    def agregar_elemento_danado(self, wait, subestacion, parque, lmt):
        """Maneja el proceso de agregar elemento dañado usando tu lógica probada"""
        try:
            self.imprimir_log("   Iniciando proceso de agregar elemento dañado...")
            
            # 1. Buscar y hacer scroll hasta el textarea usando tu lógica probada
            textarea = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, '//*[@id="collapse2"]/div/textarea'))
            )
            
            # Hacer scroll suave hasta el textarea
            self.driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", textarea)
            self.imprimir_log("   Scroll realizado hasta el textarea")
            time.sleep(5)  # Espera generosa para que termine el scroll
            
            # 2. Hacer doble clic con ActionChains (tu método probado)
            ActionChains(self.driver).double_click(textarea).perform()
            self.imprimir_log("   Doble clic realizado en textarea")
            time.sleep(15)  # Tiempo generoso para que se abra el formulario
            
            # 3. Hacer clic en el botón de agregar elemento dañado
            click_agregate = WebDriverWait(self.driver, 20).until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[2]/ul/li[2]/button/div/span'))
            )
            click_agregate.click()
            self.imprimir_log("   Botón de agregar elemento dañado clickeado")
            time.sleep(10)  # Espera para que cargue el formulario
            
            # 4. Rellenar campos del formulario
            campos_danados = [
                ('//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[2]/input', 
                 subestacion, "Subestación"),
                ('//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[4]/input', 
                 str(parque), "PARQUE"),
                ('//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[5]/input', 
                 lmt, "LMT"),
            ]
            
            for xpath, valor, nombre in campos_danados:
                try:
                    campo = wait.until(
                        EC.element_to_be_clickable((By.XPATH, xpath))
                    )
                    campo.clear()
                    campo.send_keys(valor)
                    time.sleep(2)
                    campo.send_keys(Keys.ENTER)
                    time.sleep(2)
                    self.imprimir_log(f"   Campo {nombre} rellenado: {valor}")
                except Exception as e:
                    self.imprimir_log(f"   ⚠️ No se pudo rellenar {nombre}: {str(e)}")
            time.sleep(60)
            # 5. Campo "9"
            try:
                fld9 = wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="damagedElement"]/ng-include/div/form/div[1]/ng-include/div/table/tbody/tr/td[9]/input'))
                )
                fld9.clear()
                fld9.send_keys("1")
                time.sleep(2)
                
                # Seleccionar opción del dropdown
                all_opts = wait.until(
                    EC.presence_of_all_elements_located((By.XPATH, "//ul[contains(@class,'dropdown-menu')]//a[@title]"))
                )
                
                visible_opts = [opt for opt in all_opts if opt.is_displayed()]
                priorities = ["9", "1", "9F"]
                clicked = False

                for title in priorities:
                    for opt in visible_opts:
                        if opt.get_attribute("title") == title:
                            opt.click()
                            clicked = True
                            self.imprimir_log(f"   Opción seleccionada: {title}")
                            break
                    if clicked:
                        break

                if not clicked and visible_opts:
                    visible_opts[0].click()
                    self.imprimir_log("   Primera opción visible seleccionada")
                    
            except Exception as e:
                self.imprimir_log(f"   ⚠️ No se pudo procesar campo '9': {str(e)}")
            
            # 6. Confirmaciones
            for i in range(2):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button[1]', 
                                  descripcion=f"Confirmación {i+1}.1")
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', 
                                  descripcion=f"Confirmación {i+1}.2")
                time.sleep(1)
            
            self.imprimir_log("   ✅ Elemento dañado agregado exitosamente")
            return True
            
        except Exception as e:
            self.imprimir_log(f"   ❌ Error en agregar elemento dañado: {str(e)}")
            return False

    def procesar_registro(self, index, row, wait):
        """Procesa un registro individual en una nueva pestaña"""
        ref_cc = str(row["REF CC"])
        subestacion = str(row["Subestación"])
        parque = int(row["PARQUE"])
        lmt = str(row["LMT"])

        self.imprimir_log(f"📝 Procesando registro {index + 1}: REF_CC={ref_cc}")
        
        try:
            # Abrir nueva pestaña
            self.driver.execute_script("window.open('');")
            self.driver.switch_to.window(self.driver.window_handles[-1])
            
            # INICIO LÓGICA SELENIUM EN NUEVA PESTAÑA
            self.driver.get("http://gemas-pe.enelint.global")
            time.sleep(3)

            # Navegación principal
            try:
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, "/html/body/div[3]/div/div[2]/div/div[1]/div/a/img"))
                ).click()
                time.sleep(2)

                # Buscar por REF_CC
                input_element = wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="test"]/div[1]/div[1]/div[1]/div[2]/input'))
                )
                input_element.clear()
                input_element.send_keys(ref_cc)
                time.sleep(2)
                
                # Clic en buscar
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, "/html/body/div[3]/div/div/div[2]/div/toolbar/div/div[3]/ul/li[1]/button"))
                ).click()
                time.sleep(2)

                # Seleccionar resultado
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[contains(@id, "-grid-container")]/div[2]/div'))
                ).click()
                time.sleep(2)

                # Abrir menú
                wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="menu"]/ul/li[1]/div/div[1]/a'))
                ).click()
                time.sleep(2)
                
            except Exception as e:
                raise Exception(f"Error en navegación inicial: {str(e)}")

            # Rellenar campos condicionales
            self.rellenar_condicional('//*[@id="beginRepairDate"]', '//*[@id="beginDate"]', wait)
            self.rellenar_condicional('//*[@id="localizationDate"]', '//*[@id="beginDate"]', wait)
            self.rellenar_condicional('//*[@id="endRepairDate"]', '//*[@id="endDate"]', wait)
            self.rellenar_condicional('//*[@id="estResolutionDate"]', '//*[@id="endDate"]', wait)

            # Intentar guardar múltiples veces
            for i in range(6):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', descripcion=f"Guardar {i + 1}")

            # Navegar entre pestañas y rellenar
            wait.until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[3]/div/div[2]/div/div/div/div/form/div[3]/div[2]/div/div/ul/li[1]/a'))
            ).click()
            time.sleep(2)

            for i in range(3):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button')

            # Rellenar CC
            cc = self.driver.find_element(By.XPATH, '//*[@id="textAreaCC"]/textarea')
            if not cc.get_attribute("value").strip():
                cc.send_keys(".")
            time.sleep(1)

            # Continuar con UOT
            wait.until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[3]/div/div[2]/div/div/div/div/form/div[3]/div[2]/div/div/ul/li[2]/a'))
            ).click()
            time.sleep(2)

            # Rellenar UOT
            uot = self.driver.find_element(By.XPATH, '//*[@id="textAreaUOT"]/textarea')
            if not uot.get_attribute("value").strip():
                uot.send_keys(".")
            time.sleep(1)

            # Continuar con CDS
            wait.until(
                EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div/div[1]/div[3]/div/div[2]/div/div/div/div/form/div[3]/div[2]/div/div/ul/li[3]/a'))
            ).click()
            time.sleep(2)

            # Rellenar CDS
            cds = self.driver.find_element(By.XPATH, '//*[@id="textAreaCds"]/textarea')
            if not cds.get_attribute("value").strip():
                cds.send_keys("VAut")
            time.sleep(1)

            # Guardar final
            self.intentar_click('/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[2]/ul/li[1]/button/div/img', descripcion="Guardar final")
            time.sleep(6)

            # Manejo de ventanas de confirmación
            self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button[1]', descripcion="Confirmación 1")
            time.sleep(3)
            
            self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', descripcion="Confirmación 2")
            time.sleep(3)

            # AGREGAR ELEMENTO DAÑADO (usando tu lógica probada)
            elemento_agregado = self.agregar_elemento_danado(wait, subestacion, parque, lmt)
            
            if not elemento_agregado:
                self.imprimir_log("   ⚠️ Se saltó el proceso de agregar elemento dañado debido a errores")

            # Finalizar proceso
            self.intentar_click('//*[@id="menu"]/ul/li[1]/div/div[1]/a', descripcion="Volver al menú")
            time.sleep(2)
            
            self.intentar_click('/html/body/div[3]/div/div[1]/div[2]/div/toolbar/div/div[3]/ul/li[2]/button', descripcion="Finalizar")
            time.sleep(2)

            # Confirmaciones finales múltiples
            for i in range(8):
                self.intentar_click('/html/body/div[1]/div/div/div/div[3]/button', descripcion=f"Confirmación extra {i+1}")
                time.sleep(0.5)

            # Registrar éxito
            self.resultados_proceso.append({
                "Fila": index + 1,
                "REF CC": ref_cc,
                "Subestación": subestacion,
                "PARQUE": parque,
                "LMT": lmt,
                "Estado": "✅ Correcto",
                "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "Empresa": "PLUZ ENERGÍA",
                "Elemento_Danado": "✅ Agregado" if elemento_agregado else "⚠️ No agregado"
            })
            
            self.imprimir_log(f"✅ Registro {index + 1} procesado correctamente - PLUZ")
            return True
            
        except Exception as e:
            error_msg = f"❌ Error en registro {index + 1}: {str(e)}"
            logging.error(error_msg, exc_info=True)
            self.resultados_proceso.append({
                "Fila": index + 1,
                "REF CC": ref_cc,
                "Subestación": subestacion,
                "PARQUE": parque,
                "LMT": lmt,
                "Estado": f"❌ Error: {str(e)}",
                "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "Empresa": "PLUZ ENERGÍA",
                "Elemento_Danado": "❌ Error"
            })
            self.imprimir_log(error_msg)
            return False
            
        finally:
            # Cerrar pestaña actual y volver a la principal
            try:
                if len(self.driver.window_handles) > 1:
                    self.driver.close()
                    self.driver.switch_to.window(self.driver.window_handles[0])
            except:
                pass

    def ejecutar_procesamiento(self):
        """Ejecuta el procesamiento principal"""
        try:
            # Inicializar navegador
            if not self.inicializar_navegador():
                return

            wait = WebDriverWait(self.driver, 20)

            # FASE 1: Login manual
            self.imprimir_log("🔐 FASE 1: Preparando para ingreso manual de credenciales...")
            self.imprimir_log("📝 Por favor, ingresa manualmente tu usuario y contraseña")
            self.imprimir_log("⏳ Esperando 30 segundos para ingreso manual...")
            
            # Abrir página de login
            self.driver.get("http://gemas-pe.enelint.global")
            time.sleep(40)
            
            # Verificar login
            try:
                time.sleep(5)
                current_url = self.driver.current_url
                if "login" not in current_url.lower():
                    self.imprimir_log("✅ Login manual detectado - Continuando...")
                else:
                    self.imprimir_log("⚠️ Parece que aún estás en la página de login")
            except:
                self.imprimir_log("⚠️ Continuando con procesamiento...")

            # FASE 2: Procesamiento
            self.imprimir_log("🚀 FASE 2: Iniciando procesamiento automatizado...")
            
            for index, row in self.df.iterrows():
                if not self.procesando:
                    break
                    
                self.procesar_registro(index, row, wait)
                
                # Pequeña pausa entre registros
                time.sleep(2)

            # Finalizar
            if self.procesando:
                self.imprimir_log("🎉 AUTOMATIZACIÓN COMPLETADA EXITOSAMENTE")
                self.imprimir_log("⚡ PLUZ ENERGÍA - MISIÓN CUMPLIDA ⚡")
                self.exportar_resultados()
                
        except Exception as e:
            logging.error("Error general durante la ejecución", exc_info=True)
            self.imprimir_log(f"❌ Error general: {str(e)}")
        
        finally:
            self.finalizar_procesamiento()

    def exportar_resultados(self):
        """Exporta los resultados a un archivo Excel"""
        try:
            if not self.resultados_proceso:
                self.imprimir_log("⚠️ No hay resultados para exportar")
                return
            
            df_resultados = pd.DataFrame(self.resultados_proceso)
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_path = f"resultados_proceso_{timestamp}.xlsx"
            
            df_resultados.to_excel(output_path, index=False)
            self.imprimir_log(f"📊 Resultados exportados a: {output_path}")
            
            # Resumen detallado
            correctos = len([r for r in self.resultados_proceso if 'Correcto' in str(r['Estado'])])
            errores = len([r for r in self.resultados_proceso if 'Error' in str(r['Estado'])])
            elemento_agregado = len([r for r in self.resultados_proceso if 'Agregado' in str(r.get('Elemento_Danado', ''))])
            
            self.imprimir_log(f"📈 Resumen:")
            self.imprimir_log(f"   ✅ Correctos: {correctos}")
            self.imprimir_log(f"   ❌ Errores: {errores}")
            self.imprimir_log(f"   🔧 Elemento dañado agregado: {elemento_agregado}")
            
        except Exception as e:
            self.imprimir_log(f"❌ Error al exportar resultados: {str(e)}")

    def finalizar_procesamiento(self):
        """Finaliza el procesamiento y cierra recursos"""
        try:
            if self.driver:
                self.driver.quit()
                self.imprimir_log("✅ Navegador cerrado")
        except:
            pass
        
        self.procesando = False
        self.imprimir_log("🏁 Procesamiento finalizado")

    def iniciar(self):
        """Método principal para iniciar la automatización"""
        try:
            # Cargar el archivo Excel
            if not os.path.exists(self.excel_path):
                print(f"❌ Error: Archivo Excel no encontrado en {self.excel_path}")
                return
            
            self.df = pd.read_excel(self.excel_path)
            total_filas = len(self.df)
            
            if total_filas == 0:
                print("❌ Error: El archivo Excel está vacío")
                return
            
            print(f"📊 Archivo Excel cargado: {total_filas} registros")
            print("=" * 60)
            
            # Mostrar resumen de columnas
            print("Columnas encontradas:")
            for col in self.df.columns:
                print(f"  - {col}")
            print("=" * 60)
            
            # Confirmar inicio
            respuesta = input("¿Desea iniciar el procesamiento? (s/n): ").lower()
            if respuesta != 's':
                print("Proceso cancelado")
                return
            
            self.procesando = True
            self.resultados_proceso = []
            
            # Ejecutar procesamiento
            self.ejecutar_procesamiento()
            
        except Exception as e:
            print(f"❌ Error al iniciar: {str(e)}")
        finally:
            input("\nPresione Enter para salir...")


# Ejecutar la automatización
if __name__ == "__main__":
    app = AutomatizacionGemas()
    app.iniciar()

📊 Archivo Excel cargado: 210 registros
Columnas encontradas:
  - DIA
  - REF CC
  - NT
  - UOT
  - CFT
  - PARQUE
  - Subestación
  - LMT
  - TIPO
  - CAUSA
[14:09:05] ✅ Navegador Edge inicializado correctamente
[14:09:05] 🔐 FASE 1: Preparando para ingreso manual de credenciales...
[14:09:05] 📝 Por favor, ingresa manualmente tu usuario y contraseña
[14:09:05] ⏳ Esperando 30 segundos para ingreso manual...
[14:09:50] ✅ Login manual detectado - Continuando...
[14:09:50] 🚀 FASE 2: Iniciando procesamiento automatizado...
[14:09:50] 📝 Procesando registro 1: REF_CC=B070875
